# Getting Started with OMOPy

This notebook introduces the foundational modules for working with OMOP CDM databases in Python:

- **`omopy.connector`** — Connect to a CDM database and obtain a `CdmReference`
- **`omopy.generics`** — Inspect, subset, sample, flatten, and benchmark a CDM

By the end of this notebook you will know how to:
1. Connect to a DuckDB-backed OMOP CDM
2. Explore tables and their contents
3. Query and filter clinical data
4. Take snapshots, subsets, and benchmarks of a CDM

## Configuration

Set the path to your DuckDB file and the schema name here. All subsequent cells reference these variables.

In [1]:
# -- Configuration --------------------------------------------------------
DB_PATH = "../data/synthea.duckdb"   # path to DuckDB file (relative to this notebook)
CDM_SCHEMA = "base"                  # schema containing the OMOP CDM tables
# -------------------------------------------------------------------------

## Connecting to a CDM

`cdm_from_con` is the main entry point. It accepts a database connection **or** a
string path to a DuckDB file and returns a `CdmReference` object that acts as a
handle to the entire CDM.

```python
cdm_from_con(con, *, cdm_schema=None, write_schema=None,
             cdm_version=None, cdm_name=None, cdm_tables=None)
```

In [2]:
from omopy.connector import cdm_from_con

cdm = cdm_from_con(
    DB_PATH,
    cdm_schema=CDM_SCHEMA,
)

# The repr gives a quick summary of the connection
cdm

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)

## Exploring the CDM Reference

A `CdmReference` exposes metadata about the connected CDM: its name, version,
available tables, and more.

In [3]:
# Basic metadata
print("CDM name:   ", cdm.cdm_name)
print("CDM version:", cdm.cdm_version)
print("CDM source: ", cdm.cdm_source)

CDM name:    dbt-synthea
CDM version: 5.4
CDM source:  DbSource(backend='duckdb', schema='base', version=5.4, tables=36)


In [4]:
# List every table available in this CDM
print(f"Number of tables: {len(cdm.table_names)}")
print(cdm.table_names)

Number of tables: 36
['care_site', 'cdm_source', 'concept', 'concept_ancestor', 'concept_class', 'concept_relationship', 'concept_synonym', 'condition_era', 'condition_occurrence', 'cost', 'death', 'device_exposure', 'domain', 'dose_era', 'drug_era', 'drug_exposure', 'drug_strength', 'episode', 'episode_event', 'fact_relationship', 'location', 'measurement', 'metadata', 'note', 'note_nlp', 'observation', 'observation_period', 'payer_plan_period', 'person', 'procedure_occurrence', 'provider', 'relationship', 'specimen', 'visit_detail', 'visit_occurrence', 'vocabulary']


In [5]:
# Cohort tables (if any have been created)
print("Cohort tables:", cdm.cohort_tables)

Cohort tables: {}


In [6]:
# select_tables returns a new CdmReference containing only the requested tables
clinical_cdm = cdm.select_tables(["person", "observation_period", "condition_occurrence"])
print(clinical_cdm.table_names)

['person', 'observation_period', 'condition_occurrence']


## Accessing Tables

Individual tables are accessed with **dict-style** indexing. Each table is
returned as a `CdmTable`, a thin wrapper around a lazy Ibis expression.

In [7]:
person = cdm["person"]

print("Type:      ", type(person))
print("Table name:", person.tbl_name)
print("Source:    ", person.tbl_source)
print("Columns:   ", person.columns)

Type:       <class 'omopy.generics.cdm_table.CdmTable'>
Table name: person
Source:     duckdb
Columns:    ['person_id', 'gender_concept_id', 'year_of_birth', 'month_of_birth', 'day_of_birth', 'birth_datetime', 'race_concept_id', 'ethnicity_concept_id', 'location_id', 'provider_id', 'care_site_id', 'person_source_value', 'gender_source_value', 'gender_source_concept_id', 'race_source_value', 'race_source_concept_id', 'ethnicity_source_value', 'ethnicity_source_concept_id']


In [8]:
# You can also use .get() for safe access (returns None if the table doesn't exist)
obs = cdm.get("observation_period")
print(type(obs))

<class 'omopy.generics.cdm_table.CdmTable'>


In [9]:
# .data gives the underlying Ibis Table for advanced operations
print(type(person.data))

<class 'ibis.expr.types.relations.Table'>


In [10]:
# .count() returns the number of rows
print(f"Person count: {person.count()}")

Person count: 27


In [11]:
# .schema shows the column types
person.schema

{'person_id': Int64(nullable=True),
 'gender_concept_id': Int32(nullable=True),
 'year_of_birth': Int64(nullable=True),
 'month_of_birth': Int64(nullable=True),
 'day_of_birth': Int64(nullable=True),
 'birth_datetime': Timestamp(timezone=None, scale=6, nullable=True),
 'race_concept_id': Int32(nullable=True),
 'ethnicity_concept_id': Int32(nullable=True),
 'location_id': Int32(nullable=True),
 'provider_id': Int32(nullable=True),
 'care_site_id': Int32(nullable=True),
 'person_source_value': String(length=None, nullable=True),
 'gender_source_value': String(length=None, nullable=True),
 'gender_source_concept_id': Int32(nullable=True),
 'race_source_value': String(length=None, nullable=True),
 'race_source_concept_id': Int32(nullable=True),
 'ethnicity_source_value': String(length=None, nullable=True),
 'ethnicity_source_concept_id': Int32(nullable=True)}

## Querying Data

`CdmTable` supports common data-wrangling verbs: `filter`, `select`, `head`,
and `collect`. Operations are **lazy** until you call `.collect()`, which
materialises the result as a Polars DataFrame.

In [12]:
# .head(n) — preview the first n rows (returns a CdmTable)
person.head(5).collect()

person_id,gender_concept_id,year_of_birth,month_of_birth,day_of_birth,birth_datetime,race_concept_id,ethnicity_concept_id,location_id,provider_id,care_site_id,person_source_value,gender_source_value,gender_source_concept_id,race_source_value,race_source_concept_id,ethnicity_source_value,ethnicity_source_concept_id
i64,i32,i64,i64,i64,datetime[μs],i32,i32,i32,i32,i32,str,str,i32,str,i32,str,i32
1,8532,2013,11,9,2013-11-09 00:00:00,8527,38003564,null,null,null,"""1854be9f-f085-e266-4e76-aa78df…","""F""",0,"""white""",0,"""nonhispanic""",0
2,8507,1968,9,25,1968-09-25 00:00:00,8515,38003564,null,null,null,"""1cd7cc0a-2e2a-c167-1207-cad02d…","""M""",0,"""asian""",0,"""nonhispanic""",0
3,8532,1996,6,5,1996-06-05 00:00:00,8527,38003564,null,null,null,"""1d3d16ce-a3a4-5e9e-1b4d-58a391…","""F""",0,"""white""",0,"""nonhispanic""",0
4,8507,1973,10,28,1973-10-28 00:00:00,8527,38003564,null,null,null,"""261ad2a4-7c6f-da54-6a5b-3441be…","""M""",0,"""white""",0,"""nonhispanic""",0
5,8507,1973,10,28,1973-10-28 00:00:00,8527,38003564,null,null,null,"""280ee8b5-6ebb-11cf-e4db-8cf101…","""M""",0,"""white""",0,"""nonhispanic""",0


In [13]:
# .select(*cols) — keep only the listed columns
person.select("person_id", "year_of_birth", "gender_concept_id").head(5).collect()

person_id,year_of_birth,gender_concept_id
i64,i64,i32
1,2013,8532
2,1968,8507
3,1996,8532
4,1973,8507
5,1973,8507


In [14]:
# .filter(expr) — filter rows using an Ibis expression built from .data
young = person.filter(person.data.year_of_birth > 1990)
print(f"Persons born after 1990: {young.count()}")
young.select("person_id", "year_of_birth").head(5).collect()

Persons born after 1990: 9


person_id,year_of_birth
i64,i64
1,2013
3,1996
7,2013
10,2014
11,2010


In [15]:
# .collect() materialises the full (possibly filtered) table as a Polars DataFrame
df = person.select("person_id", "year_of_birth").collect()
print(type(df))
print(df.shape)
df.head()

<class 'polars.dataframe.frame.DataFrame'>
(27, 2)


person_id,year_of_birth
i64,i64
1,2013
2,1968
3,1996
4,1973
5,1973


## CDM Snapshot

`snapshot()` produces a high-level summary of the CDM as a `CdmSnapshot`
pydantic model. It includes person counts, observation period date ranges,
vocabulary version, and more.

In [16]:
from omopy.connector import snapshot

snap = snapshot(cdm)
snap

CdmSnapshot(cdm_name='dbt-synthea', cdm_source_name='dbt-synthea', cdm_description='An OMOP CDM derived from a Synthea dataset using dbt.', cdm_documentation_reference='https://synthetichealth.github.io/synthea/', cdm_version='5.4', cdm_holder='OHDSI', cdm_release_date='2024-10-06 00:00:00', vocabulary_version='v5.0 29-FEB-24', person_count=27, observation_period_count=27, earliest_observation_period_start_date=datetime.date(1968, 1, 21), latest_observation_period_end_date=datetime.date(2024, 2, 2), snapshot_date='2026-03-30')

In [17]:
# Key fields on the CdmSnapshot
print("CDM name:          ", snap.cdm_name)
print("CDM source name:   ", snap.cdm_source_name)
print("CDM version:       ", snap.cdm_version)
print("Person count:      ", snap.person_count)
print("Obs period count:  ", snap.observation_period_count)
print("Vocabulary version:", snap.vocabulary_version)
print("Earliest obs start:", snap.earliest_observation_period_start_date)
print("Latest obs end:    ", snap.latest_observation_period_end_date)
print("Snapshot date:     ", snap.snapshot_date)

CDM name:           dbt-synthea
CDM source name:    dbt-synthea
CDM version:        5.4
Person count:       27
Obs period count:   27
Vocabulary version: v5.0 29-FEB-24
Earliest obs start: 1968-01-21
Latest obs end:     2024-02-02
Snapshot date:      2026-03-30


In [18]:
# Convert to a Polars DataFrame or a plain dict
snap.to_polars()

cdm_name,cdm_source_name,cdm_description,cdm_documentation_reference,cdm_version,cdm_holder,cdm_release_date,vocabulary_version,person_count,observation_period_count,earliest_observation_period_start_date,latest_observation_period_end_date,snapshot_date
str,str,str,str,str,str,str,str,str,str,str,str,str
"""dbt-synthea""","""dbt-synthea""","""An OMOP CDM derived from a Syn…","""https://synthetichealth.github…","""5.4""","""OHDSI""","""2024-10-06 00:00:00""","""v5.0 29-FEB-24""","""27""","""27""","""1968-01-21""","""2024-02-02""","""2026-03-30"""


In [19]:
snap.to_dict()

{'cdm_name': 'dbt-synthea',
 'cdm_source_name': 'dbt-synthea',
 'cdm_description': 'An OMOP CDM derived from a Synthea dataset using dbt.',
 'cdm_documentation_reference': 'https://synthetichealth.github.io/synthea/',
 'cdm_version': '5.4',
 'cdm_holder': 'OHDSI',
 'cdm_release_date': '2024-10-06 00:00:00',
 'vocabulary_version': 'v5.0 29-FEB-24',
 'person_count': '27',
 'observation_period_count': '27',
 'earliest_observation_period_start_date': '1968-01-21',
 'latest_observation_period_end_date': '2024-02-02',
 'snapshot_date': '2026-03-30'}

## Subsetting the CDM

Two helpers let you create a smaller CDM for development or testing:

- **`cdm_subset(cdm, person_ids)`** — keep only the specified person IDs
- **`cdm_sample(cdm, n, *, seed=None)`** — randomly sample *n* persons

In [20]:
from omopy.connector import cdm_subset, cdm_sample

# Grab a few person IDs to subset on
some_ids = person.select("person_id").head(10).collect()["person_id"].to_list()
print("Person IDs for subset:", some_ids)

Person IDs for subset: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [21]:
# cdm_subset keeps only the rows for the given person_ids across all tables
small_cdm = cdm_subset(cdm, some_ids)
print(small_cdm)
print(f"Person count in subset: {small_cdm['person'].count()}")

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)
Person count in subset: 10


In [22]:
# cdm_sample randomly selects n persons (reproducible with seed)
sampled_cdm = cdm_sample(cdm, 50, seed=42)
print(sampled_cdm)
print(f"Person count in sample: {sampled_cdm['person'].count()}")

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)
Person count in sample: 27


## Flattening the CDM

`cdm_flatten` joins all clinical tables onto `person` and returns a single
wide Ibis Table. Call `.to_polars()` to materialise the result.

In [23]:
from omopy.connector import cdm_flatten

# Flatten on the sampled CDM to keep things small
flat = cdm_flatten(sampled_cdm)
print(type(flat))  # ibis Table

<class 'ibis.expr.types.relations.Table'>


In [24]:
# Materialise to Polars
flat_df = flat.to_polars()
print(f"Shape: {flat_df.shape}")
flat_df.head()

Shape: (866, 8)


person_id,observation_concept_id,start_date,end_date,type_concept_id,domain,observation_concept_name,type_concept_name
i64,i32,date,date,i32,str,str,str
25,19075601,2003-02-23,2004-02-29,32838,"""drug_exposure""","""clopidogrel 75 MG Oral Tablet""","""EHR prescription"""
21,4311629,1991-11-27,null,32827,"""condition_occurrence""","""Impaired glucose tolerance""","""EHR encounter record"""
6,1539463,2013-01-02,2014-01-02,32838,"""drug_exposure""","""simvastatin 10 MG Oral Tablet""","""EHR prescription"""
24,40231918,2024-01-14,2024-01-14,32838,"""drug_exposure""","""acetaminophen 325 MG / oxycodo…","""EHR prescription"""
21,19080128,1994-12-14,1995-12-20,32838,"""drug_exposure""","""lisinopril 10 MG Oral Tablet""","""EHR prescription"""


## Benchmarking

`benchmark` runs a standard set of tasks against the CDM and reports timing
information as a Polars DataFrame.

In [25]:
from omopy.connector import benchmark

bench = benchmark(cdm)
print(type(bench))  # polars DataFrame
print("Columns:", bench.columns)
bench

<class 'polars.dataframe.frame.DataFrame'>
Columns: ['task', 'time_taken_secs', 'time_taken_mins', 'dbms', 'person_n']


task,time_taken_secs,time_taken_mins,dbms,person_n
str,f64,f64,str,i32
"""distinct count of concept rela…",0.0039,0.0001,"""duckdb""",27
"""count of different relationshi…",0.0022,0.0,"""duckdb""",27
"""join of concept and concept cl…",0.0606,0.001,"""duckdb""",27
"""concept table collected into m…",0.0079,0.0001,"""duckdb""",27
"""join of person and observation…",0.0051,0.0001,"""duckdb""",27
"""summary of observation period …",0.0037,0.0001,"""duckdb""",27


## Working with Core Types

OMOPy ships several core data types. Here we briefly introduce `Codelist`
and `SummarisedResult`; they will be covered in depth in later notebooks.

### Codelist

A `Codelist` is a dict subclass that maps concept-set names to lists of
OMOP concept IDs.

In [26]:
from omopy.generics import Codelist

cl = Codelist({
    "diabetes": [201826, 4193704],
    "hypertension": [316866, 4028741],
})

print("Names:          ", cl.names)
print("All concept IDs:", cl.all_concept_ids)
print("Keys:           ", list(cl.keys()))
print("Values:         ", list(cl.values()))

Names:           ['diabetes', 'hypertension']
All concept IDs: {4193704, 316866, 201826, 4028741}
Keys:            ['diabetes', 'hypertension']
Values:          [[201826, 4193704], [316866, 4028741]]


In [27]:
# Iterate just like a normal dict
for name, ids in cl.items():
    print(f"  {name}: {ids}")

  diabetes: [201826, 4193704]
  hypertension: [316866, 4028741]


### SummarisedResult (preview)

`SummarisedResult` is the standard container for analytics outputs in OMOPy.
It wraps a Polars DataFrame and provides helpers like `.tidy()`,
`.pivot_estimates()`, `.filter_settings()`, `.suppress()`, and `.split_group()`.

We will use it extensively in notebook 02 when running characterisation and
incidence analyses.

## Summary

In this notebook we covered the core workflow:

| Step | Function / Object | Purpose |
|------|-------------------|---------|
| Connect | `cdm_from_con()` | Obtain a `CdmReference` |
| Explore | `.table_names`, `.cdm_name`, `.cdm_version` | Inspect metadata |
| Access | `cdm["table"]` | Get a lazy `CdmTable` |
| Query | `.filter()`, `.select()`, `.head()`, `.collect()` | Wrangle data |
| Snapshot | `snapshot(cdm)` | High-level CDM summary |
| Subset | `cdm_subset()`, `cdm_sample()` | Create smaller CDMs |
| Flatten | `cdm_flatten()` | Join all tables into one |
| Benchmark | `benchmark()` | Performance profiling |
| Types | `Codelist`, `SummarisedResult` | Core data structures |

**Next steps:** Continue to [02_characterisation.ipynb](./02_characterisation.ipynb)
to learn about patient-level characterisation, cohort creation, and incidence
analyses.